In [1]:
from openai import OpenAI
import sys
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import fitz
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

In [4]:
import re

pdf_path = (
    r"D:\Ai engineering course\Projects\RAG_Lyrics_Generator"
    r"\Declan McKenna - Slipping Through My Fingers (Official Audio).pdf"
)

doc = fitz.open(pdf_path)

text = ""

for page in doc:
    text += page.get_text() + "\n"

doc.close()



pattern = (
    r"\[(\d{2}:\d{2}:\d{2}\.\d{2})\]\s*-\s*"
    r"(.*?)(?=\[\d{2}:\d{2}:\d{2}\.\d{2}\]|$)"
)

matches = re.findall(pattern, text, re.DOTALL)

segments = []


def timestamp_to_seconds(timestamp):

    hours, minutes, seconds = timestamp.split(":")
    return (
        int(hours) * 3600
        + int(minutes) * 60
        + float(seconds)
    )


for timestamp, content in matches:

    content = " ".join(content.split())

    speaker_match = re.match(
        r"(Speaker\s+\d+)\s+(.*)",
        content
    )

    if speaker_match:
        speaker = speaker_match.group(1)
        lyric_text = speaker_match.group(2)
    else:
        speaker = None
        lyric_text = content

    segments.append({
        "timestamp": timestamp,
        "start_time": timestamp_to_seconds(timestamp),
        "speaker": speaker,
        "text": lyric_text
    })



chunk_size = 300

chunks = []

current_segments = []
current_length = 0


for segment in segments:

    segment_length = len(segment["text"])


    if (
        current_length + segment_length > chunk_size
        and current_segments
    ):

        chunks.append({
            "chunk_id": len(chunks),
            "start_time": current_segments[0]["start_time"],
            "end_time": current_segments[-1]["start_time"],
            "text": " ".join(
                item["text"]
                for item in current_segments
            ),
            "segments": current_segments
        })

        current_segments = []
        current_length = 0

    current_segments.append(segment)
    current_length += segment_length



if current_segments:

    chunks.append({
        "chunk_id": len(chunks),
        "start_time": current_segments[0]["start_time"],
        "end_time": current_segments[-1]["start_time"],
        "text": " ".join(
            item["text"]
            for item in current_segments
        ),
        "segments": current_segments
    })



for chunk in chunks:

    print(f"\n{'=' * 60}")
    print(f"CHUNK {chunk['chunk_id']}")
    print(f"Start: {chunk['start_time']} seconds")
    print(f"End:   {chunk['end_time']} seconds")

    print("\nText:")
    print(chunk["text"])

    print("\nSegments:")

    for segment in chunk["segments"]:

        print(
            f"[{segment['timestamp']}] "
            f"{segment['speaker']}: "
            f"{segment['text']}"
        )


CHUNK 0
Start: 2.12 seconds
End:   34.03 seconds

Text:
School bag in hand, she leaves home in the early morning, waving goodbye with an absent-minded smile. I watch her go with a surge of that well-known sadness. And I have to sit down for a while. The feeling that I'm losing her forever while never really entering her world.

Segments:
[00:00:02.12] Speaker 1: School bag in hand, she leaves home in the early morning, waving goodbye with an absent-minded smile. I watch her go with a surge of that well-known sadness. And I have to sit down for a while.
[00:00:34.03] Speaker 1: The feeling that I'm losing her forever while never really entering her world.

CHUNK 1
Start: 49.18 seconds
End:   61.11 seconds

Text:
I'm glad whenever I can share Her laughter, that funny little girl. Slipping through my fingers all the time. I try to catch her every minute.

Segments:
[00:00:49.18] Speaker 1: I'm glad whenever I can share Her laughter, that funny little girl.
[00:01:01.11] Speaker 1: Slippi

In [12]:
texts = [chunk["text"] for chunk in chunks]


vectorizer = TfidfVectorizer()

index = vectorizer.fit_transform(texts)



def find_lyric_time(query, k=3):

    query_vector = vectorizer.transform([query])

    scores = index @ query_vector.T
    scores = scores.toarray().flatten()

    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for i in top_indices:

        results.append({
            "chunk_id": chunks[i]["chunk_id"],
            "start_time": chunks[i]["start_time"],
            "end_time": chunks[i]["end_time"],
            "text": chunks[i]["text"],
            "score": scores[i]
        })

    return results



def format_time(seconds):

    minutes = int(seconds // 60)
    seconds = int(seconds % 60)

    return f"{minutes:02d}:{seconds:02d}"





In [16]:
query = "slipping through my fingers"

results = find_lyric_time(query, k=3)

for result in results:

    print(f"\nTimestamp: {format_time(result['start_time'])}")
    print(f"Score: {result['score']:.3f}")
    print(f"Text: {result['text']}")


Timestamp: 02:34
Score: 0.560
Text: Slipping through my fingers all the time, I try to capture every minute, the feeling in it. Slipping through my fingers all the time, do I really see what's in her mind? Each time I think I'm close to knowing. She keeps on growing, slipping through my fingers all the time.

Timestamp: 01:10
Score: 0.364
Text: The feeling in it, slipping through my fingers all the time. Do I really see what's in her mind each time I think? I'm close to knowing. She keeps on growing, slipping through my fingers all the time. Sleep in our eyes, her and me at the breakfast table. Barely awake, I let precious time go by.

Timestamp: 00:49
Score: 0.308
Text: I'm glad whenever I can share Her laughter, that funny little girl. Slipping through my fingers all the time. I try to catch her every minute.
